<a href="https://colab.research.google.com/github/ashwathraj/DataScience_AI/blob/main/Cross_Validation_Diabetes_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Load the diabetes dataset
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
display(X.head())

Feature matrix shape: (442, 10)
Target variable shape: (442,)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [2]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
dtypes: float64(10)
memory usage: 34.7 KB


### Implementing Cross-Validation with Linear Regression

We will create a pipeline containing scaling and a `LinearRegression` model, then evaluate it using 5-Fold Cross-Validation using the $R^2$ scoring metric.

In [3]:
X.describe()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02
mean,-2.511817e-19,1.230790e-17,-2.245564e-16,-4.797570e-17,-1.381499e-17,3.918434e-17,-5.777179e-18,-9.042540e-18,9.293722e-17,1.130318e-17
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01


In [5]:
y.head()

,target
0,151.0
1,75.0
2,141.0
3,206.0
4,135.0


In [6]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


### Train-Test Split & Performance Evaluation

We will perform an 80:20 train-test split, scale the features, train our `LinearRegression` model, and compute the following training metrics:
* **$R^2$ Score**
* **Root Mean Squared Error (RMSE)**
* **Mean Squared Error (MSE)**
* **Mean Absolute Error (MAE)**
* **Mean Absolute Percentage Error (MAPE)**

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

# 1. Split the dataset into train and test sets (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Build and fit the pipeline on the training data
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipeline.fit(X_train, y_train)

# 3. Predict on the training dataset
y_train_pred = pipeline.predict(X_train)

# 4. Calculate metrics on the training dataset
train_r2 = r2_score(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mape = mean_absolute_percentage_error(y_train, y_train_pred)

# Calculate Adjusted R2
n = X_train.shape[0]
p = X_train.shape[1]
train_adj_r2 = 1 - ((1 - train_r2) * (n - 1) / (n - p - 1))

# 5. Display the metrics
metrics_df = pd.DataFrame({
    'Metric': ['R2 Score', 'Adjusted R2', 'RMSE', 'MSE', 'MAE', 'MAPE'],
    'Train Value': [train_r2, train_adj_r2, train_rmse, train_mse, train_mae, train_mape]
})

display(metrics_df)

,Metric,Train Value
0,R2 Score,0.527919
1,Adjusted R2,0.514116
2,RMSE,53.558843
3,MSE,2868.549703
4,MAE,43.483504
5,MAPE,0.389199


### Performance Evaluation on Test Dataset

Now we will apply the trained pipeline to the test dataset and compute the same metrics to evaluate its generalization ability:
* **$R^2$ Score**
* **Adjusted $R^2$**
* **Root Mean Squared Error (RMSE)**
* **Mean Squared Error (MSE)**
* **Mean Absolute Error (MAE)**
* **Mean Absolute Percentage Error (MAPE)**

In [15]:
# 1. Predict on the test dataset using the trained pipeline
y_test_pred = pipeline.predict(X_test)

# 2. Calculate metrics on the test dataset
test_r2 = r2_score(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mape = mean_absolute_percentage_error(y_test, y_test_pred)

# Calculate Adjusted R2 for test dataset
n_test = X_test.shape[0]
p_test = X_test.shape[1]
test_adj_r2 = 1 - ((1 - test_r2) * (n_test - 1) / (n_test - p_test - 1))

# 3. Display the test metrics in a table
test_metrics_df = pd.DataFrame({
    'Metric': ['R2 Score', 'Adjusted R2', 'RMSE', 'MSE', 'MAE', 'MAPE'],
    'Test Value': [test_r2, test_adj_r2, test_rmse, test_mse, test_mae, test_mape]
})

display(test_metrics_df)

,Metric,Test Value
0,R2 Score,0.452603
1,Adjusted R2,0.382424
2,RMSE,53.853446
3,MSE,2900.193628
4,MAE,42.794095
5,MAPE,0.374998


### 5-Fold Cross-Validation & Side-by-Side Comparison

We will now perform 5-fold cross-validation on the entire dataset using Mean Squared Error (MSE) as the metric.

We will then compare the metrics side-by-side:
1. **Train Value** (Evaluated on the 80% training set)
2. **Test Value** (Evaluated on the 20% validation/test set)
3. **Cross-Validation (CV) Value** (Average across the 5 folds)

In [16]:
# 1. Set up 5-fold cross-validation
kf_compare = KFold(n_splits=5, shuffle=True, random_state=42)

# 2. Compute CV scores for R2 and MSE
# Note: cross_val_score returns negative MSE for optimization purposes; we negate it to get the actual MSE
cv_mse_scores = -cross_val_score(pipeline, X, y, cv=kf_compare, scoring='neg_mean_squared_error')
cv_r2_scores = cross_val_score(pipeline, X, y, cv=kf_compare, scoring='r2')

# Calculate mean CV metrics
cv_mean_mse = cv_mse_scores.mean()
cv_mean_rmse = np.sqrt(cv_mse_scores).mean()
cv_mean_r2 = cv_r2_scores.mean()

# For Adjusted R2 under 5-fold CV, we use the average number of samples per validation fold
n_cv = len(y) * (4/5)
p_cv = X.shape[1]
cv_mean_adj_r2 = 1 - ((1 - cv_mean_r2) * (n_cv - 1) / (n_cv - p_cv - 1))

# Compute other CV metrics (MAE, MAPE) for comparison completeness
cv_mae_scores = -cross_val_score(pipeline, X, y, cv=kf_compare, scoring='neg_mean_absolute_error')
cv_mape_scores = -cross_val_score(pipeline, X, y, cv=kf_compare, scoring='neg_mean_absolute_percentage_error')

cv_mean_mae = cv_mae_scores.mean()
cv_mean_mape = cv_mape_scores.mean()

# 3. Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Metric': ['R2 Score', 'Adjusted R2', 'RMSE', 'MSE', 'MAE', 'MAPE'],
    'Train Value': [train_r2, train_adj_r2, train_rmse, train_mse, train_mae, train_mape],
    'Test Value': [test_r2, test_adj_r2, test_rmse, test_mse, test_mae, test_mape],
    '5-Fold CV Mean Value': [cv_mean_r2, cv_mean_adj_r2, cv_mean_rmse, cv_mean_mse, cv_mean_mae, cv_mean_mape]
})

display(comparison_df)

,Metric,Train Value,Test Value,5-Fold CV Mean Value
0,R2 Score,0.527919,0.452603,0.478470
1,Adjusted R2,0.514116,0.382424,0.463248
2,RMSE,53.558843,53.853446,54.848941
3,MSE,2868.549703,2900.193628,3015.381694
4,MAE,43.483504,42.794095,44.269746
5,MAPE,0.389199,0.374998,0.395392


### 5-Fold Cross-Validation: Linear Regression vs. Random Forest

We will perform a 5-fold cross-validation on our training dataset (`X_train`, `y_train`) using both a `LinearRegression` pipeline and a `RandomForestRegressor` pipeline. We will display the Mean Squared Error (MSE) of each individual fold side-by-side.

In [21]:
from sklearn.ensemble import RandomForestRegressor

# 1. Define pipelines for both estimators
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(random_state=42))
])

# 2. Set up cross-validation split strategy
kf_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Calculate MSE for each individual fold on the training set
# We negate the scores because cross_val_score returns negative MSE by default
lr_mse_folds = -cross_val_score(lr_pipeline, X_train, y_train, cv=kf_cv, scoring='neg_mean_squared_error')
rf_mse_folds = -cross_val_score(rf_pipeline, X_train, y_train, cv=kf_cv, scoring='neg_mean_squared_error')

# 4. Create comparison DataFrame showing fold-by-fold MSE
fold_comparison_df = pd.DataFrame({
    'Fold': [f'Fold {i+1}' for i in range(5)],
    'Linear Regression MSE': lr_mse_folds,
    'Random Forest MSE': rf_mse_folds
})

# Append a row for average performance
average_row = pd.DataFrame({
    'Fold': ['Average (Mean)'],
    'Linear Regression MSE': [lr_mse_folds.mean()],
    'Random Forest MSE': [rf_mse_folds.mean()]
})

fold_comparison_df = pd.concat([fold_comparison_df, average_row], ignore_index=True)
display(fold_comparison_df)

,Fold,Linear Regression MSE,Random Forest MSE
0,Fold 1,2848.750700,3155.700006
1,Fold 2,3221.810992,3851.100569
2,Fold 3,2813.572352,3537.410139
3,Fold 4,2965.895577,3428.439883
4,Fold 5,3520.680697,3742.185731
5,Average (Mean),3074.142064,3542.967266


### Training MSE Comparison: Linear Regression vs. Random Forest

We will fit both pipelines on the entire training set (`X_train`, `y_train`) and directly calculate and compare their resulting training Mean Squared Errors (MSE).

In [20]:
# 1. Fit both models on the training dataset
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# 2. Predict on the training dataset
y_train_pred_lr = lr_pipeline.predict(X_train)
y_train_pred_rf = rf_pipeline.predict(X_train)

# 3. Calculate MSE on the training dataset
train_mse_lr = mean_squared_error(y_train, y_train_pred_lr)
train_mse_rf = mean_squared_error(y_train, y_train_pred_rf)

# 4. Display the comparison
train_comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Training MSE': [train_mse_lr, train_mse_rf]
})

display(train_comparison_df)

,Model,Training MSE
0,Linear Regression,2868.549703
1,Random Forest,460.291586
